In [3]:
!pip install -q faiss-cpu sentence-transformers transformers

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline, AutoTokenizer

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Creating knowledge base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = model.encode(
    kb,
    convert_to_numpy=True,
    show_progress_bar=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created!")

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

Creating knowledge base...
Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (13 outputs left out)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
result = zs(
    prompt_150,
    candidate_labels=labels_150
)

score = result["scores"][result["labels"].index(ans_150)]

print(round(score,3))

0.384


In [5]:
query = model.encode([prompt_150], convert_to_numpy=True)

D, I = index.search(query, 10)

retrieved_indices = I[0]

print(retrieved_indices)

rank = np.where(retrieved_indices == 150)[0]

if len(rank):
    print(rank[0] + 1)
else:
    print("Not in Top 10")

[ 663 1701 1269 1532  576  847 1693 1906  168  150]
10


In [6]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

scores = cross_encoder.predict(pairs)

reranked = np.argsort(scores)[::-1]

reranked_indices = retrieved_indices[reranked]

print(reranked_indices)

rank = np.where(reranked_indices == 150)[0][0] + 1

print(rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[ 150 1906  847 1693 1269 1532  168  576 1701  663]
1


In [7]:
row42 = train.iloc[42]

prompt42 = str(row42["prompt"])

query = model.encode([prompt42], convert_to_numpy=True)

_, I = index.search(query, 5)

docs = [kb[i] for i in I[0]]

context = " ".join(docs)

rag = f"Context: {context} Question: {prompt42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer(
    rag,
    truncation=False
)

print(len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [8]:
true_doc = kb[150]

rag = f"Context: {true_doc} Question: {prompt_150}"

result = zs(
    rag,
    candidate_labels=labels_150
)

score = result["scores"][result["labels"].index(ans_150)]

print(round(score,3))

0.989


In [9]:
bad_doc = kb[999]

rag = f"Context: {bad_doc} Question: {prompt_150}"

result = zs(
    rag,
    candidate_labels=labels_150
)

score = result["scores"][result["labels"].index(ans_150)]

print(round(score,3))

0.529


In [10]:
hits = 0

for i in range(100):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    correct = str(row[row["answer"]])

    emb = model.encode([prompt], convert_to_numpy=True)

    _, I = index.search(emb,5)

    docs = [kb[j] for j in I[0]]

    if any(correct in doc for doc in docs):
        hits += 1

hit_rate = hits / 100 * 100

print(round(hit_rate,1))

73.0


In [11]:
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    score = 0.0

    num_hits = 0.0

    for i,p in enumerate(predicted):

        if p == actual and p not in predicted[:i]:
            num_hits += 1
            score += num_hits/(i+1)

    return score

scores = []

letters = ["A","B","C","D","E"]

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])

    emb = model.encode([prompt], convert_to_numpy=True)

    _, I = index.search(emb,5)

    docs = [kb[j] for j in I[0]]

    pairs = [[prompt,d] for d in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    rag = f"Context: {best_doc} Question: {prompt}"

    options = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    result = zs(
        rag,
        candidate_labels=options
    )

    predicted_letters = []

    for label in result["labels"]:

        predicted_letters.append(
            letters[options.index(label)]
        )

    scores.append(
        apk(
            row["answer"],
            predicted_letters,
            3
        )
    )

print(round(np.mean(scores),3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.975
